# Security Configuration

Secure your ADEPT deployment with best practices.

## Network Access Note (Opt-In Required)

This operator notebook queries Keycloak's OpenID configuration directly.
By default, JupyterLab is isolated on `frontend_net` and cannot reach Keycloak.

The Keycloak well-known endpoint IS accessible via nginx_proxy (no override needed):
`https://nginx_proxy/realms/master/.well-known/openid-configuration`

For direct Keycloak admin access, add JupyterLab to `application_net`:

```yaml
# deployment/local/docker-compose/docker-compose.override.yaml
services:
  jupyterlab:
    networks:
      - adept_frontend_network
      - adept_application_network  # Required for direct Keycloak access
```

Then rebuild: `make rebuild-jupyterlab`

## Security Topics
- TLS/SSL configuration
- Authentication with Keycloak
- RBAC policies
- Network policies
- Secret management

## Docker Configuration
- JupyterLab service: `deployment/local/docker-compose/docker-compose.core.yaml`
- Rebuild after config changes: `make rebuild-jupyterlab`

In [ ]:
import os
import json
import httpx

# Load config
config = json.load(open("/tmp/adept_config.json"))
NGINX_URL = config.get('nginx_url', 'https://nginx_proxy')

async def check_keycloak_config():
    """Check Keycloak OIDC configuration via nginx_proxy (always accessible from frontend_net)."""
    url = f"{NGINX_URL}/realms/master/.well-known/openid-configuration"
    
    async with httpx.AsyncClient(verify=False, timeout=10.0) as client:
        response = await client.get(url)
        config_data = response.json()
        
        print("Keycloak Configuration (via nginx_proxy):")
        print(f"Issuer: {config_data['issuer']}")
        print(f"Auth Endpoint: {config_data['authorization_endpoint']}")
        print(f"Token Endpoint: {config_data['token_endpoint']}")
        print(f"\nSupported Grant Types:")
        for grant in config_data['grant_types_supported'][:5]:
            print(f"  - {grant}")
        
        print(f"\nNote: Issuer is '{config_data['issuer']}'")
        print("This is the nginx_proxy-routed issuer accepted by all ADEPT services.")

await check_keycloak_config()

## Network Policies

In [ ]:
# Kubernetes NetworkPolicy example
network_policy = '''
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: agent-gateway-policy
spec:
  podSelector:
    matchLabels:
      app: agent-gateway
  policyTypes:
  - Ingress
  - Egress
  ingress:
  - from:
    - podSelector:
        matchLabels:
          app: nginx-proxy
    ports:
    - protocol: TCP
      port: 8081
  egress:
  - to:
    - podSelector:
        matchLabels:
          app: orchestration-service
    ports:
    - protocol: TCP
      port: 8084
'''

print(network_policy)